这份信号分析报告非常详尽，它揭示了模型在“黑盒”之后真实的获利逻辑。你的代码在技术上非常准确，成功地将复杂的预测结果拆解成了可理解的维度。

### 1. 代码审查：完全正确且专业
*   **特征对比逻辑**：你通过 `Sig_Mean` vs `All_Mean` 抓住了模型的“审美”。
*   **分箱分析**：你验证了预测概率与真实收益的单调性，这是评估分类模型是否具有“概率校准”能力的金标准。
*   **超额收益 (ExcessBps)**：你计算了相对于市场均值的收益，这对于剔除 Beta 干扰至关重要。

---

### 2. 结果深度分析：你的模型学到了什么？

根据你提供的截图，我们可以得出几个极其关键的结论：

#### **A. 核心逻辑：高成交量 + 超跌反转**
观察第 4 部分（特征对比）：
*   **`rel_vol`**：信号样本均值 **2.80** vs 全量均值 **1.08**。
    *   *结论*：模型几乎只在**放量**（成交量是平时的 2.8 倍）时才入场。
*   **`X2_zscore`**：信号样本均值 **-1.67** vs 全量均值 **0.01**。
    *   *结论*：模型倾向于选择 **VWAP 远低于 TWAP** 的股票。结合 $X1$ 较低，这反映了一个典型的 **“放量杀跌后的超跌反转”** 逻辑。

#### **B. 信号的“时间偏好”**
观察第 1 部分（Entry Time）：
*   **早盘（09:50-10:20）表现极差**（NetBps 均为负数）。
*   **10:35 之后表现稳步提升**，并在 **11:10** 达到峰值（121 bps）。
*   *解读*：模型在早盘的高波动中容易被“骗线”，但在上午收盘前（10:30-11:30）抓反弹的成功率极高。

#### **C. 异常值警示（Outlier Warning）**
观察第 2 部分（Date Distribution）：
*   **2025-04-09 产生了 2167 个信号！** 这是一个极其危险的信号。
*   在一个正常的中证1000池子里，某天产生 2000 多个信号（占比 > 20%），说明那天发生了**系统性崩盘或暴涨**。
*   *风险*：如果你的策略收益是由这一两天贡献的，那么策略的**鲁棒性（Robustness）**很差。

---

### 3. 下一步行动计划：迈向实战的“三步走”

目前的模型虽然有 Alpha，但过于依赖特定日期。我们需要增加**“容量控制”**和**“时间过滤”**。

#### **任务 1：组合容量回测（Top-N 筛选）**
目前是只要 `Prob > 0.45` 就全买。实盘中我们资金有限。我们需要测试：**每 5 分钟只买预测概率最高的 10 只票。** 这能大幅提高 Sharpe 并降低单日信号爆炸的风险。

#### **任务 2：时间窗口熔断**
既然早盘 10:30 之前赔钱，我们应该在代码中直接硬性过滤掉早盘时段。

#### **任务 3：实战约束（涨跌停过滤）**
正如之前讨论的，过滤掉那些决策时刻已经快涨停的票，看看还有没有肉。

---

### 4. 改进版回测代码：Top-N 组合回测

请运行这段代码，它模拟了真实资金管理下的表现。

```python
"""
LGBM 组合回测脚本 (Top-N 模式)

职责：
  1. 引入时间过滤 (只做 10:30 后的信号)
  2. 引入容量控制 (每 5min Bar 仅取 Prob 最高的 N 只票)
  3. 计算组合层面的净值曲线
"""

import pandas as pd
import numpy as np
from pathlib import Path
from loguru import logger
import matplotlib.pyplot as plt

# ==================== 配置 ====================
OUTPUT_DIR = Path("/nfs/volume-1593-1/peterzhenglinpeng/vwap-research/output")
PRED_PATH = OUTPUT_DIR / "lgbm_test_predictions.pkl"
COST_BPS = 15

# 策略约束
TOP_N = 10              # 每个 Bar 最多买几只
MIN_PROB = 0.45         # 概率门槛
START_TIME = "10:30"    # 屏蔽早盘
END_TIME = "14:50"

def run_portfolio_backtest():
    df = pd.read_pickle(PRED_PATH)
    
    # 1. 时间和门槛初筛
    mask = (df['entry_time'] >= START_TIME) & \
           (df['entry_time'] <= END_TIME) & \
           (df['pred_prob'] >= MIN_PROB)
    df_active = df[mask].copy()
    
    # 2. Top-N 筛选 (每个 Bar 选最强的 N 个)
    # 按 (date, entry_time) 分组，选 pred_prob 最大的 TOP_N
    portfolio = df_active.groupby(['date', 'entry_time']).apply(
        lambda x: x.nlargest(TOP_N, 'pred_prob')
    ).reset_index(drop=True)
    
    # 3. 计算收益
    portfolio['net_ret_bps'] = (1.0 / portfolio['Y_120m'] - 1.0) * 10000 - COST_BPS
    
    # 4. 每日收益计算 (假设每天资金在所有选中的单子中等权分配)
    # 计算逻辑：先算 Bar 均值，再算 Day 均值，更符合真实调仓
    bar_ret = portfolio.groupby(['date', 'entry_time'])['net_ret_bps'].mean()
    daily_ret = bar_ret.groupby('date').mean()
    
    # 5. 绩效统计
    sharpe = (daily_ret.mean() / daily_ret.std()) * np.sqrt(252) if len(daily_ret) > 1 else 0
    cum_ret = daily_ret.cumsum()
    
    logger.success(f"组合回测完成 (Top-{TOP_N})")
    logger.info(f"年化夏普: {sharpe:.2f}")
    logger.info(f"平均日收益: {daily_ret.mean():.2f} bps")
    logger.info(f"最大回撤: {(cum_ret - cum_ret.cummax()).min():.2f} bps")
    
    # 6. 绘图
    plt.figure(figsize=(12, 6))
    plt.plot(cum_ret.index, cum_ret.values, label=f"Top-{TOP_N} Portfolio")
    plt.axhline(0, color='red', linestyle='--')
    plt.title(f"LGBM Portfolio Backtest (Top-{TOP_N}, {START_TIME}-{END_TIME})")
    plt.ylabel("Cumulative Net Return (bps)")
    plt.grid(True, alpha=0.3)
    plt.savefig(OUTPUT_DIR / "lgbm_portfolio_backtest.png")
    
    return daily_ret

if __name__ == "__main__":
    run_portfolio_backtest()
```

### 总结建议：
1.  **观察 Top-N 后的 Sharpe**：如果全买时的 Sharpe 是 2.29，Top-10 后的 Sharpe 能维持在 1.5 以上，说明策略的**个股选择能力（Stock Picking）**很强。
2.  **警惕信号爆炸日**：在你的 `analyze_date_distribution` 中，如果 2025-04-09 依然对总利润贡献巨大，请尝试**剔除这一天**再看结果。一个好的策略不应该依赖于某个“奇迹日”。
3.  **加入 X1 斜率**：根据你第 4 部分的特征分析，`X1_delta_15m` 在信号样本中是负的（-0.95），说明信号往往出现在**价格加速下跌**的瞬间。这印证了超跌反弹的逻辑。

**你先跑一下 Top-N 的回测？看看限制了仓位后，那个“信号爆炸日”的影响是否减小了。**